In [1]:
import numpy as np
import pandas as pd
import mh_toolbox.base.eurobench_collection as ec
import pathlib
import copy
from loguru import logger

pattern = r"subject_(?P<subject>\w+)_cond_(?P<condition>\w+)_run_(?P<run>\w+)"

# This route has to be changed depending on the subject to process
route_root_experiments = '/Users/jorge/BasesDatos/NEUROMARK/'
# Orden requerido por URJC
var_order = ['l_elbow_flexion', 'r_elbow_flexion', 'l_elbow_pronation', 'r_elbow_pronation', 'l_elbow_deviation',
             'r_elbow_deviation',
             'l_knee_flexion', 'r_knee_flexion', 'l_knee_adduction', 'r_knee_adduction', 'l_knee_rotation',
             'r_knee_rotation',
             'l_ankle_flexion', 'r_ankle_flexion', 'l_ankle_adduction', 'r_ankle_adduction', 'l_ankle_rotation',
             'r_ankle_rotation',
             'l_hip_rotation', 'r_hip_rotation', 'l_hip_adduction', 'r_hip_adduction', 'l_hip_flexion',
             'r_hip_flexion',
             'l_wrist_flexion', 'r_wrist_flexion', 'l_wrist_pronation', 'r_wrist_pronation', 'l_wrist_deviation',
             'r_wrist_deviation',
             'l_shoulder_flexion', 'r_shoulder_flexion', 'l_shoulder_adduction', 'r_shoulder_adduction',
             'l_shoulder_rotation', 'r_shoulder_rotation',
             'l_t4_shoulder_flexion', 'r_t4_shoulder_flexion', 'l_t4_shoulder_adduction', 'r_t4_shoulder_adduction',
             'l_t4_shoulder_rotation', 'r_t4_shoulder_rotation',
             'l_ballfoot_flexion', 'r_ballfoot_flexion', 'l_ballfoot_adduction', 'r_ballfoot_adduction',
             'l_ballfoot_rotation', 'r_ballfoot_rotation',
             'l5s1_flexion', 'l5s1_lateralFlexion', 'l5s1_axialFlexion',
             'l4l3_flexion', 'l4l3_lateralFlexion', 'l4l3_rotation',
             't9t8_flexion', 't9t8_lateralFlexion', 't9t8_rotation',
             'l1t12_flexion', 'l1t12_lateralFlexion', 'l1t12_rotation',
             't1c7_flexion', 't1c7_rotation', 't1c7_lateralFlexion',
             'head_flexion', 'head_lateralFlexion', 'head_rotation',
             ]

# subject = 'SUBJECT_27'
session = 'SESSION_01'
# condition = '02'
experiment_in = 'EUROBENCH_DATA'
results_out_results = pathlib.Path.cwd().joinpath('MOTION_HUB_RESULTS')
results_out_results.mkdir(parents=True, exist_ok=True)

n_samples = 101

list_subject = [
    '15', '16', '17', '18', '23', '24', 
    '25', '26', '29', '30', '31',
    '33', '34', '35', '36', '37', '38',
    '40', '42', '43', '44', '45']  #'27','20','39','32',
list_subject = ['15']

# list_subject = ['39'] -> run_04: da error
# list_condition = ['03'] -> run_04: da error
# Processing file subject_32_cond_05_run_04_jointAngles.csv da error, el mismo error para subject_45_cond_05_run_04
# 'Error finding the indices to remove, nullify or divide using the tuple: Initial event 9.300000190734863 is greater than fina...

list_condition = ['01', '02', '03', '04', '05']  #

# Subject 05 -> sin datos XSens
for curr_condition in list_condition:
    print(f'Processing condition {curr_condition}')
    for subject in list_subject:
        curr_subject = f'SUBJECT_{subject}'
        print(f'Processing subject {curr_subject}')
        
        route_in_data = pathlib.Path(route_root_experiments).joinpath(experiment_in, curr_subject, session, 'IRREGULAR_TERRAIN').joinpath('XSENS')
        print(f'Route In Eurobench data: {route_in_data}')
        
        route_in_vicon = pathlib.Path(route_root_experiments).joinpath(experiment_in, curr_subject, session, 'IRREGULAR_TERRAIN').joinpath('VICON')
        print(f'Route In Vicon data: {route_in_vicon}')
    
        # Out route for the figures
        route_results = pathlib.Path(results_out_results).joinpath(curr_subject)
        route_figures_individual = route_results.joinpath('figures_xsens_resampled').joinpath('condition_' + curr_condition)
        route_figures_average = route_results.joinpath('figures_xsens_resampled_average').joinpath('condition_' + curr_condition)
        route_figures_all = route_results.joinpath('figures_xsens_resampled_all').joinpath('condition_' + curr_condition)
        print(f'Route Figures Individual: {route_figures_individual}')
        print(f'Route Figures Average: {route_figures_average}')
        print(f'Route Figures All: {route_figures_all}')
    
        # create a glob pattern to find all the files that match the pattern subject_\d+_cond_01_run_\d+_jointAngles.csv
        pattern = f'**/subject_*_cond_{curr_condition}_run_*_jointAngles.csv'
        files = list(pathlib.Path(route_in_data).glob(pattern))
        print(files)
        
        eu_collection = ec.EurobenchCollection()
        eu_collection.initialize_from_dir_eurobench_data(list_eurobench_data=files,
                                                         route_companion_element=route_in_vicon,
                                                         load_files=True)
        full_data_class = eu_collection.extract_data_class(inner_nest_by='file', outer_nest_by='segment')
        full_data_class.invert_data_class_nesting()
        full_data_class.plot_dataclass(bool_show=False, bool_save=True, bool_events=True,
                                       save_directory=route_figures_all)
        
        # -- Split eliminating the general_start_turn and general_end_turn events
        eu_collection.split_collection_using_events(events_key='events', 
                                         events_category=('general_start_turn', 'general_end_turn'), 
                                         id_suffix='divide', 
                                         tolerance=0.1)
        
        # ------ Split using the l_heel_strike events        
        left_collection = copy.deepcopy(eu_collection)
        left_collection.curate_collection(cols_to_filter=['!r_' ], 
                                          list_reorder_cols_data=var_order)
        left_collection.split_collection_using_events(events_key='events', 
                                         events_category='l_heel_strike', 
                                         id_suffix='split_by_strike', 
                                         tolerance=0.1)
        left_collection.resample_collection(method='n_samples', n_samples=n_samples)
        dict_left_data_class = left_collection.extract_data_class(inner_nest_by='file', outer_nest_by='segment')
        dict_left_data_class.invert_data_class_nesting()
        
        # ----- Split using the l_heel_strike events        
        right_collection = copy.deepcopy(eu_collection)
        right_collection.curate_collection(cols_to_filter=['r_', 'time'],
                                          list_reorder_cols_data=var_order)
        right_collection.split_collection_using_events(events_key='events', 
                                         events_category='r_heel_strike', 
                                         id_suffix='split_by_strike', 
                                         tolerance=0.1)
        right_collection.resample_collection(method='n_samples', n_samples=n_samples)
        dict_right_data_class = right_collection.extract_data_class(inner_nest_by='file', outer_nest_by='segment')
        dict_right_data_class.invert_data_class_nesting()
        
        # Combine
        dict_left_data_class.combine_data_class(dict_right_data_class)
        has_data = dict_left_data_class.has_data_data_class()
        
        dict_left_data_class.plot_dataclass(bool_show=False, bool_save=True, bool_events=True,
                                             save_directory=route_figures_individual)
        print('Plots done')
        
        dict_left_data_class.plot_dataclass_average(bool_show=False,
                                                    bool_std=True,
                                                    bool_save=True,
                                                    save_directory=route_figures_average)
        
        # Data ala URJC. According to URJC requirements
        dataframes_averages = []
        dataframes_all = {}
        for key in dict_left_data_class.data_class.keys():
            logger.info(f"key: {key}")
    
            if key == 'time':
                continue
    
            for key_inner, val_inner in dict_left_data_class.data_class[key].outer_nest.items():
                logger.info(f"key_inner: {key_inner}")
    
                if key_inner not in dataframes_all:
                    dataframes_all[key_inner] = {}
                if key not in dataframes_all[key_inner]:
                    dataframes_all[key_inner][key] = []
                # Append the data to the list
                dataframes_all[key_inner][key].extend(val_inner.data)
    
            # Average values is a list, create a dataframe from it
            average_values, _ = dict_left_data_class.data_class[key].get_statistics_outer_nest()
            dataframes_averages.append(pd.DataFrame(average_values, columns=[key]))
        
        # Concatenate all the dataframes in the list
        df_average = pd.concat(dataframes_averages, axis=1)
        # Reorder the columns, the elements of var_order first, then the rest
        df_average = df_average[var_order + [col for col in df_average.columns if col not in var_order]]
        
        # Save the dataframe in a xls file
        route_data_average = pathlib.Path(route_results).joinpath('data_average')
        pathlib.Path.mkdir(route_data_average, parents=True, exist_ok=True)
        df_average.to_excel(route_data_average.
                            joinpath(f'subject_{curr_subject}_{session}_condition_{curr_condition}.xlsx'))
        
        # Save in a xls file a sheet for each key_inner
        route_data_individual_xls = pathlib.Path(route_results).joinpath('data_individual_xls')
        pathlib.Path.mkdir(route_data_individual_xls, parents=True, exist_ok=True)
    
        for key, val_all in dataframes_all.items():
            full_route = route_data_individual_xls.joinpath(
                f'{key[:-4]}.xlsx')
            print(full_route)
            with pd.ExcelWriter(full_route) as writer:
                print(key)
                for key_inner, val_inner in val_all.items():
                    print(key_inner)
                    column_names = [key] * len(val_inner)
                    df = pd.DataFrame(np.array(val_inner).T)
                    df.to_excel(writer, sheet_name=key_inner, index=False, header=False)
        
        print('Saved')

Processing condition 01
Processing subject SUBJECT_15
Route In Eurobench data: /Users/jorge/BasesDatos/NEUROMARK/EUROBENCH_DATA/SUBJECT_15/SESSION_01/IRREGULAR_TERRAIN/XSENS
Route In Vicon data: /Users/jorge/BasesDatos/NEUROMARK/EUROBENCH_DATA/SUBJECT_15/SESSION_01/IRREGULAR_TERRAIN/VICON
Route Figures Individual: /Users/jorge/Documents/Coding/Projects/Misiones/motion-hub-toolbox/docs/tutorials/neuromark/MOTION_HUB_RESULTS/SUBJECT_15/figures_xsens_resampled/condition_01
Route Figures Average: /Users/jorge/Documents/Coding/Projects/Misiones/motion-hub-toolbox/docs/tutorials/neuromark/MOTION_HUB_RESULTS/SUBJECT_15/figures_xsens_resampled_average/condition_01
Route Figures All: /Users/jorge/Documents/Coding/Projects/Misiones/motion-hub-toolbox/docs/tutorials/neuromark/MOTION_HUB_RESULTS/SUBJECT_15/figures_xsens_resampled_all/condition_01
[]


AttributeError: 'EurobenchCollection' object has no attribute 'initialize_from_dir_eurobench_data'